# ML-02 — Research Question and Provisional Lane
**Imteshar Ahmed · Week 1 · 14 October 2026**

[Open in Colab](https://colab.research.google.com/github/iayamin6/FlyRank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb)

## 1. My lane (or freestyle) and why
**Provisional lane 2: Refresh / Content Opportunity Scoring.** I want to help an editor decide which existing pages deserve inspection first when review time is limited. This builds on my completed starter notebooks and my Python/ML background, while giving me a concrete decision-support output rather than a model without a user. The starter data contains exposure, freshness and observed movement signals that can support a transparent queue. I will start with a rule, audit it, and only add ML if it improves honest validation. The three descriptive results below motivate spending the next seven weeks on this lane; they do not establish that refreshing pages works. I can revise the lane through the end of Week 4.

## 2. The question: decision, action, cost of a wrong call
**Search question:** Among pages with enough measured search exposure, which should a content editor review first for a possible refresh or continued monitoring, and can an evidence-based ranking prioritize them more usefully than a simple freshness rule?

| Part | Provisional definition |
|---|---|
| Decision and user | A content editor chooses the next 20 pages to inspect from an eligible inventory. Twenty is a proposed review budget, not an observed staffing fact. |
| Unit of analysis | One pseudonymized content item belonging to a client at a decision cutoff. The starter has one snapshot row per item with trailing-90-day measurements; it is not a daily panel. Later, one item–cutoff pair will be used for time-based evaluation. |
| Output | A ranked top-20 review queue with pseudonymous IDs, score, exposure/freshness reason codes, data-quality flags, uncertainty, and a suggested **review** action. |
| Action | The editor checks page intent, relevance, freshness and related-page performance, then chooses refresh, protect, or monitor. The anonymized data cannot itself support a content rewrite; no automated edits, pruning or publication. |
| False positive cost | Wasted review/editing time and possible harm from changing an already useful page; therefore review comes before intervention. |
| False negative cost | A page with a real search problem receives attention late, potentially allowing missed visibility or clicks to persist. No monetary loss is estimated without business evidence. |

**Why data/ML might help:** exposure, age and movement can disagree, and a fixed stale-page rule may over-prioritize old but stable content. Aggregating measurements makes a large inventory reviewable. A model must earn its added complexity; a transparent rule or analysis is a valid final result if a model does not improve it.

**Baseline and evaluation, specified before modeling:** initially require at least 500 impressions over 90 days and rank by days since last update, descending, then impressions descending. This is an illustrative volume policy, not a universal threshold; later compare 300/500/1,000 and K=10/20/50 using development data. The code computes descriptive precision@20 against the starter's `trend_direction == "down"` proxy. This proxy is a rule-derived summary of already observed movement, **not** ground truth about refresh benefit or future decline.

For the capstone, the proposed observable outcome is next-30-day impressions relative to the preceding 30 days, with adequate coverage and a minimum prior-window volume. I will define a decline cutoff and persistence checks in the data contract before evaluating models. Primary metric: precision@20 for this future observed-decline event versus the frozen rule on the same eligible set; also report base rate, recall, client-level variation and uncertainty. That operational event still does not identify pages that would benefit from an edit. Features must end at the decision cutoff; no target-window totals or current metadata that leak later updates. This future evaluation **cannot be computed from the starter snapshot alone**.

**One-paragraph frame:** For a content editor deciding which pages to inspect next, I will build an explainable review queue from exposure and freshness signals, evaluating whether it concentrates later observed search declines more effectively than a simple rule. A wrong recommendation wastes attention or misses a problem. Data may help reconcile competing signals, but ML is optional. I will claim only observed associations and validated decision-support performance, not the causal benefit of a refresh.

## 3. Quick look at the data (2–3 real numbers)
Run all cells in order. The loader uses the repository CSV locally, or the identical version pinned to the already published starter-work commit when run in a fresh Colab session. It prints only aggregate results, never raw pages or private text. Dependencies: pandas and the standard library; no warehouse credentials are required.

In [1]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
import hashlib
import platform
import pandas as pd

RELATIVE_DATA = Path("data/raw/content_refresh_anonymized.csv")
STARTER_COMMIT = "9d66094c89e765b88bb709694ca4e6f545e5318e"
DATA_URL = f"https://raw.githubusercontent.com/iayamin6/FlyRank-ML-Internship/{STARTER_COMMIT}/{RELATIVE_DATA.as_posix()}"
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / RELATIVE_DATA).is_file()), None)
if root is not None:
    raw = (root / RELATIVE_DATA).read_bytes()
    source = RELATIVE_DATA.as_posix()
else:
    with urlopen(DATA_URL, timeout=60) as response:
        raw = response.read()
    source = f"pinned public starter at {STARTER_COMMIT[:7]}"
df = pd.read_csv(BytesIO(raw))
required = {"content_id", "client_id", "impressions_90d", "days_since_last_update", "trend_direction"}
assert required.issubset(df.columns)
assert df["content_id"].notna().all() and df["content_id"].is_unique
assert df["client_id"].notna().all()
assert df["impressions_90d"].ge(0).all()
print(f"Source: {source}")
print(f"SHA-256: {hashlib.sha256(raw).hexdigest()}")
print(f"Python {platform.python_version()} | pandas {pd.__version__}")
print(f"Grain verified: {len(df):,} unique content items, {df.client_id.nunique()} clients, {df.shape[1]} columns.")

Source: data/raw/content_refresh_anonymized.csv
SHA-256: c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396
Python 3.13.0 | pandas 3.0.6
Grain verified: 30,000 unique content items, 32 clients, 44 columns.


In [2]:
MIN_IMPRESSIONS = 500
STALE_DAYS = 90
K = 20
visible = df["impressions_90d"].ge(MIN_IMPRESSIONS)
declining_proxy = df["trend_direction"].eq("down")
stale = df["days_since_last_update"].ge(STALE_DAYS)
summary = pd.DataFrame([
    {"Observation": "Pages with >=500 impressions", "Pages": int(visible.sum())},
    {"Observation": "Pages with >=500 impressions and observed down trend", "Pages": int((visible & declining_proxy).sum())},
    {"Observation": "Pages with >=500 impressions and >=90 days since update", "Pages": int((visible & stale).sum())},
])
summary["Percent of all pages"] = (100 * summary["Pages"] / len(df)).round(2)
print(summary.to_string(index=False))

                                            Observation  Pages  Percent of all pages
                           Pages with >=500 impressions  16726                 55.75
   Pages with >=500 impressions and observed down trend   9961                 33.20
Pages with >=500 impressions and >=90 days since update   6575                 21.92


**What these three numbers suggest:**

1. **16,726 of 30,000 pages (55.75%)** meet the provisional 500-impression floor. There is a substantial inventory in which to prioritize review, but the floor alone does not establish statistical reliability.
2. **9,961 pages (33.20% of all pages)** meet that floor and have a recorded downward trend. This suggests an attention-allocation problem worth investigating; a downward trend can still reflect seasonality, consolidation or noise.
3. **6,575 pages (21.92% of all pages)** meet the floor and have not been updated for at least 90 days. Freshness can therefore support a simple, inspectable baseline. Staleness is not proof of poor quality.

The groups overlap and must not be added together. All percentages use all 30,000 starter rows as their denominator. The 500-impression and 90-day thresholds were chosen as provisional decision policies, not optimized against model performance.

In [3]:
# Metric feasibility check only: no model fitting and no held-out performance claim.
# Missing update age is excluded, not silently interpreted as fresh or stale.
eligible = df.loc[visible & df["days_since_last_update"].notna()].copy()
assert len(eligible) >= K
queue = eligible.sort_values(
    ["days_since_last_update", "impressions_90d", "content_id"],
    ascending=[False, False, True],
).head(K)
hits = int(queue["trend_direction"].eq("down").sum())
proxy_precision = hits / K
proxy_base_rate = eligible["trend_direction"].eq("down").mean()
print(f"Eligible baseline pool: {len(eligible):,} pages")
print(f"Observed down-trend proxy in top {K}: {hits}/{K} ({proxy_precision:.1%})")
print(f"Same proxy in eligible pool: {proxy_base_rate:.1%}")
print("Descriptive snapshot check only; not forecast accuracy or refresh-effect evidence.")

Eligible baseline pool: 16,726 pages
Observed down-trend proxy in top 20: 16/20 (80.0%)
Same proxy in eligible pool: 59.6%
Descriptive snapshot check only; not forecast accuracy or refresh-effect evidence.


The baseline's **16/20 (80.0%)** observed-down-trend share, versus **59.6%** in its eligible pool, shows that precision@20 is computable for the teaching proxy. It does not demonstrate generalization: the queue and proxy use the same snapshot, the top 20 may be concentrated in a few clients, and there is no future test window. I will not tune against this result or call it a successful predictive model.

## 4. Careful words: what I can and can't claim
**Can claim now:** I observed the aggregate counts above in this starter slice, verified its page-level grain, and computed a descriptive baseline diagnostic. The lane is provisional and the queue is for human review.

**Cannot claim now:** that age causes decline; that editing these pages will recover traffic or revenue; that the baseline predicts future outcomes; that this sample represents all clients or the warehouse; or that I discovered a Google ranking factor. No article text is available to judge content quality directly.

**Validation and data safeguards for the next seven weeks:**
- Start with a data contract and signal audit. Inspect missingness by content type; use explicit availability flags rather than blind zero imputation. Treat position zero as missing. Rate columns are percentages (0.76 means 0.76%); scroll/AI rates can exceed 100 because of their measurement definitions.
- Exclude `trend_direction`, `trend_pct`, IDs and any rule-derived priority flags from model predictors. For future labels, also exclude all aggregates overlapping the outcome window; dropping the two trend fields alone is insufficient. IDs serve joins, splits and a deterministic tie-break only.
- Before warehouse modeling, verify grain, date range and each client's tracking coverage; require availability flags `IS TRUE`. Work on a middle-month partition, such as March 2026, while keeping the final June 2026 window sealed for the final test. Use time separation plus client holdout checks, and group repeated page observations appropriately.
- Define future outcomes from measured window totals, then check persistence, volume, client-wide movement and related-page changes. A decline label remains an operational definition, not a product decision or a causal treatment effect.
- Compare the frozen rule and any model on identical held-out candidates; report precision@20, recall, pool base rate, by-client results and uncertainty. Inspect the top 20 for explainable reasons and data-quality problems. Keep the simpler method if added complexity does not show credible benefit.
- Produce only safe aggregate evidence and pseudonymous queues; never expose raw client names, domains, private queries or credentials. An authorized editor must inspect actual content before acting.

**Plan:** Weeks 2–3: contract, signal audit and coverage checks; Week 4: confirm or revise the lane and freeze baseline/metrics; Weeks 5–6: optional model and leakage/time/client validation; Weeks 7–8: human-review playbook and careful public write-up. These are planned steps, not completed results.

**References used:** repository `docs/ml-intern-dataset-and-lane-guide.md` (lane 2, thresholds, validation); `docs/data-dictionary.md`; `skills/framing-ml-problems/SKILL.md`; `skills/flyrank/flyrank-data/SKILL.md`.

## 5. Self-check
- [x] One predefined lane selected provisionally, with a reason.
- [x] Decision, user, unit, output, action and error costs named.
- [x] Three supporting starter-data counts computed, with denominators and interpretation.
- [x] Baseline and metric specified; proxy diagnostic clearly separated from future validation.
- [x] No causal or guaranteed-recovery claims; ML must earn its place.
- [x] Notebook executed top to bottom without errors; only aggregate outputs retained.
- [x] Saved for submission at `work/notebooks/w01_research_question.ipynb` in my repository.

AI assisted the framing and code. Supporting numbers are computed from the starter CSV; the editor's capacity and thresholds are explicit provisional assumptions, not facts about an actual team's workflow.